# Replicating Macro Figures from the RSPG Paper

This notebook replicates Figures 2 (panel 3), 3, and 10 from
[*Recurrent Structural Policy Gradient for Partially Observable Mean Field Games*](https://arxiv.org/abs/2602.20141)
for the **Macroeconomics** environment, using SPG and RSPG.

Requires a **GPU runtime** on Colab.

In [ ]:
# --- Setup: clone repo and install dependencies ---
!git clone https://github.com/ben-moll/mfax.git /content/mfax 2>/dev/null || echo "repo already cloned"
import sys; sys.path.insert(0, "/content/mfax")
!pip install -q -r /content/mfax/dev/requirements.txt

import jax
print(f"JAX devices: {jax.devices()}")

In [ ]:
# --- Train SPG and RSPG on the Macroeconomics environment ---
from mfax.train import train

overrides = dict(
    task="endogenous", state_type="states", discount_factor=0.95,
    num_envs=8, num_iterations=200, eval_frequency=20, lr=0.01,
    common_noise=True, normalize_obs=True, normalize_states=True, seed=0,
)

results = {}
for algo in ["spg", "rspg"]:
    results[algo] = train(algo, max_time=300, **overrides)

## Figure 2, Panel 3: Exploitability vs Training Time (Macroeconomics)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for algo, style in [("spg", {"color": "C0", "label": "SPG"}),
                     ("rspg", {"color": "C1", "label": "RSPG"})]:
    res = results[algo]
    ax.plot(res.train_times, res.exploitabilities, **style)
ax.set_yscale("log")
ax.set_xlabel("Train Time (s)")
ax.set_ylabel("Exploitability")
ax.set_title("Macroeconomics")
ax.legend()
plt.tight_layout()
plt.show()

## Figure 3: Mean-Field Distribution Heatmaps (Macroeconomics)

Rows: SPG, RSPG. Columns: interest rate, wage, MF distribution at t=0, t=64, t=127.

In [ ]:
import numpy as np
import matplotlib.colors as mcolors

fig, axes = plt.subplots(2, 5, figsize=(22, 6))

for row, algo in enumerate(["spg", "rspg"]):
    seq = results[algo].final_eval_results.mf_sequence
    agg = seq.aggregate_s

    # interest rate and wage over time (first env)
    r = np.array(agg.interest_rate[:, 0])
    w = np.array(agg.wage[:, 0])
    axes[row, 0].plot(r)
    axes[row, 1].plot(w)
    if row == 0:
        axes[row, 0].set_title("Interest rate")
        axes[row, 1].set_title("Wage")
    axes[row, 0].set_ylabel(algo.upper())

    # MF distribution heatmaps at selected timesteps
    mu = np.array(agg.mu[:, 0])          # (T, 1000)
    n_wealth, n_income = 200, 5
    mu_2d = mu.reshape(mu.shape[0], n_wealth, n_income)  # (T, wealth, income)
    vmax = mu_2d.max()

    for col, t in enumerate([0, 64, 127]):
        im = axes[row, 2 + col].imshow(
            mu_2d[t].T, aspect="auto", origin="lower",
            norm=mcolors.LogNorm(vmin=max(1e-6, mu_2d[t].max() * 1e-4), vmax=vmax),
        )
        if row == 0:
            axes[row, 2 + col].set_title(f"t = {t}")
        axes[row, 2 + col].set_xlabel("Wealth")
        if col == 0:
            axes[row, 2 + col].set_ylabel("Income")

plt.tight_layout()
plt.show()

## Figure 10: Detailed Macro Policy Plots

Rows: value function, expected action, consumption, reward. Columns: SPG, RSPG.
Lines colored by income level.

In [ ]:
from mfax.envs import make_env as _make_env
_env = _make_env("pushforward/endogenous", common_noise=True)

n_wealth, n_income = 200, 5
wealth_grid = np.array(_env.params.states[:n_wealth, 0])   # first n_wealth states, wealth dim
income_grid = np.array(_env.params.states[::n_wealth, 1])  # every n_wealth-th state, income dim
discrete_actions = np.array(_env.params.discrete_actions)

row_labels = ["Value function", "Expected action", "Consumption", "Reward"]
fig, axes = plt.subplots(4, 2, figsize=(12, 16))

for col, algo in enumerate(["spg", "rspg"]):
    ev = results[algo].final_eval_results
    seq = ev.mf_sequence

    # shapes: (num_envs, n_states) and (T, num_envs, n_states, n_actions)
    disc_ret = np.array(ev.policy_disc_returns[0]).reshape(n_wealth, n_income)
    prob_a_t0 = np.array(seq.prob_a[0, 0])          # (1000, 20)
    mat_r_t0 = np.array(seq.mat_r[0, 0])            # (1000, 20)
    r_t0 = float(np.array(seq.aggregate_s.interest_rate[0, 0]))
    w_t0 = float(np.array(seq.aggregate_s.wage[0, 0]))

    exp_action = (prob_a_t0 * discrete_actions[None, :]).sum(axis=-1).reshape(n_wealth, n_income)
    exp_reward = (prob_a_t0 * mat_r_t0).sum(axis=-1).reshape(n_wealth, n_income)
    # consumption = fraction_consumed * budget
    budget = (1 + r_t0) * wealth_grid[:, None] + w_t0 * income_grid[None, :]
    consumption = exp_action * budget

    data = [disc_ret, exp_action, consumption, exp_reward]
    for row_idx, (d, label) in enumerate(zip(data, row_labels)):
        for inc_idx in range(n_income):
            axes[row_idx, col].plot(
                wealth_grid, d[:, inc_idx],
                label=f"y={income_grid[inc_idx]:.2f}" if col == 0 else None,
            )
        if col == 0:
            axes[row_idx, col].set_ylabel(label)
        axes[row_idx, col].set_xlabel("Wealth")
    axes[0, col].set_title(algo.upper())

axes[0, 0].legend(fontsize=7, title="Income")
plt.tight_layout()
plt.show()